In [1]:
!pip install pyspark==3.3.2
# !java -version
# import sys
# print(sys.executable)
# print(sys.version)

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("exploracao_patry").getOrCreate()

import sys
sys.path.insert(0, ".")
from exploracao_v3 import run_synthesis_from_tables

# Carrega as tabelas via Spark
instrumento = spark.read.options(header=True, inferSchema=True, sep=",").csv("data_csv/instrumento.csv")
operacao    = spark.read.options(header=True, inferSchema=True, sep=",").csv("data_csv/operacao.csv")
tipo_if     = spark.read.options(header=True, inferSchema=True, sep=",").csv("data_csv/tipo_if.csv")

tables = {
    "tipo_if":     tipo_if,
    "instrumento": instrumento,
    "operacao":    operacao,
}

# Specs: PKs e relacionamentos entre as tabelas
specs_config = {
    "tipo_if": {
        "pk_cols": ["NUM_TIPO_IF"],
        "static": True,  
    },
    "instrumento": {
        "pk_cols": ["NUM_IF"],
        "foreign_keys": [
            {
                "columns":        ["NUM_TIPO_IF"],
                "parent_table":   "tipo_if",
                "parent_columns": ["NUM_TIPO_IF"],
            }
        ],
    },
    "operacao": {
        "pk_cols": ["NUM_ID_OPERACAO"],
        "foreign_keys": [
            {
            "columns":        ["NUM_IF", "COD_IF"],
            "parent_table":   "instrumento",
            "parent_columns": ["NUM_IF", "COD_IF"],
            }
        ],
    },
}

# Executa a sintetização
synthetic = run_synthesis_from_tables(
    tables=tables,
    specs_config=specs_config,
    verbose=True,
    scale_factor=10,
)

# Exibe amostras dos dados sintéticos gerados
for nome, df in synthetic.items():
    print(f"\n=== {nome} ===")
    df.show(2, truncate=False)


c:\ProgramData\anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\ProgramData\anaconda3\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
c:\ProgramData\anaconda3\lib\site-packages\numpy\.libs\libopenblas64__v0.3.23-246-g3d31191b-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


Ordem topológica: tipo_if -> instrumento -> operacao
n_rows_by_table: {'tipo_if': 5, 'instrumento': 10000, 'operacao': 500000}
Specs ativas após saneamento de relacionamentos:
  tipo_if: sem FK ativa
  OK: instrumento.['NUM_TIPO_IF'] -> tipo_if.['NUM_TIPO_IF']
  OK: operacao.['NUM_IF', 'COD_IF'] -> instrumento.['NUM_IF', 'COD_IF']
[tipo_if] STATIC | 5 linhas
[instrumento] PAI | 1000->10000
[operacao] FILHO | 50000->500000
Validando...
Validação OK.

=== tipo_if ===
+-----------+-----------+--------------------------------+-------------------+-------------------+
|NUM_TIPO_IF|COD_TIPO_IF|NOM_TIPO_IF                     |DAT_INCLUSAO       |DAT_ALTERACAO      |
+-----------+-----------+--------------------------------+-------------------+-------------------+
|1          |CDB        |Certificado de Depósito Bancário|2019-12-01 00:00:00|2019-12-01 00:00:00|
|2          |LCI        |Letra de Crédito Imobiliário    |2019-12-01 00:00:00|2019-12-01 00:00:00|
+-----------+-----------+----------

In [3]:
tipo_if.show(10,False)
instrumento.filter(F.col('num_if').isin([384,260,694,70])).show(10,False)
operacao.show(4,False)

+-----------+-----------+--------------------------------------+-------------------+-------------------+
|NUM_TIPO_IF|COD_TIPO_IF|NOM_TIPO_IF                           |DAT_INCLUSAO       |DAT_ALTERACAO      |
+-----------+-----------+--------------------------------------+-------------------+-------------------+
|1          |CDB        |Certificado de Depósito Bancário      |2019-12-01 00:00:00|2019-12-01 00:00:00|
|2          |LCI        |Letra de Crédito Imobiliário          |2019-12-01 00:00:00|2019-12-01 00:00:00|
|3          |LCA        |Letra de Crédito do Agronegócio       |2019-12-01 00:00:00|2019-12-01 00:00:00|
|4          |LF         |Letra Financeira                      |2019-12-01 00:00:00|2019-12-01 00:00:00|
|5          |DPGE       |Depósito a Prazo com Garantia Especial|2019-12-01 00:00:00|2019-12-01 00:00:00|
+-----------+-----------+--------------------------------------+-------------------+-------------------+

+------+---------+------------+-----------+-----------

In [9]:
synthetic['tipo_if'].show(10,False)
synthetic['instrumento'].filter(F.col('num_if').isin([1001,1002,1003,1004])).show(10,False)
synthetic['operacao'].filter(F.col('num_if').isin([1001,1002,1003,1004])).show(10,False)


+-----------+-----------+--------------------------------------+-------------------+-------------------+
|NUM_TIPO_IF|COD_TIPO_IF|NOM_TIPO_IF                           |DAT_INCLUSAO       |DAT_ALTERACAO      |
+-----------+-----------+--------------------------------------+-------------------+-------------------+
|1          |CDB        |Certificado de Depósito Bancário      |2019-12-01 00:00:00|2019-12-01 00:00:00|
|2          |LCI        |Letra de Crédito Imobiliário          |2019-12-01 00:00:00|2019-12-01 00:00:00|
|3          |LCA        |Letra de Crédito do Agronegócio       |2019-12-01 00:00:00|2019-12-01 00:00:00|
|4          |LF         |Letra Financeira                      |2019-12-01 00:00:00|2019-12-01 00:00:00|
|5          |DPGE       |Depósito a Prazo com Garantia Especial|2019-12-01 00:00:00|2019-12-01 00:00:00|
+-----------+-----------+--------------------------------------+-------------------+-------------------+

+------+---------+------------+-----------+-----------